# SNP Selection — EA Smoking Status (DoubleML + Stability Selection, Corrected)

**Purpose:** Identify SNPs causally associated with smoking_status via cross-fitted
DoubleML residualization + stability selection. Mirrors AA's 03_snp_selection_smoking.ipynb.

**Inputs:**
- `checkpoint7_snp_encoded_012.csv`, `confounders_X.npy` (corrected, full SVD)
- `checkpoint2_metadata_sample_filtered.csv`

**Key parameters:** p < 0.001 per-repeat, 30 repeats, ≥80% stability threshold,
5-fold cross-fitting — identical to AA.

In [1]:
import pandas as pd
import numpy as np
import re
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto = X_snp_first[keep_mask].T.astype(np.float64)

p = X_auto.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_snp_mask = denom > 1e-8
X_standardized = (X_auto[:, valid_snp_mask] - 2 * p[valid_snp_mask]) / denom[valid_snp_mask]

probe_ids_valid = probe_id_array[keep_mask][valid_snp_mask]
print("X_standardized shape:", X_standardized.shape)

X_confounders = np.load(os.path.join(out_dir, "confounders_X_relatedness_filtered.npy"))
print("Confounders shape:", X_confounders.shape)

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
status_map = meta_df.set_index("sample_id")["smoking_status"]
Y = np.array([1.0 if status_map.get(sid) == "Smoker" else 0.0 for sid in sample_ids])
print("Y distribution:", np.unique(Y, return_counts=True))

X_standardized shape: (1460, 127416)
Confounders shape: (1460, 12)
Y distribution: (array([0., 1.]), array([665, 795]))


In [2]:
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))

    return theta, pvals

print("Single test iteration...")
start = time.time()
theta_test, pvals_test = doubleml_scan(X_standardized, Y, X_confounders, random_state=0)
print(f"Done in {time.time()-start:.1f}s")
for thresh in [0.01, 0.001, 0.0001]:
    print(f"p < {thresh}: {(pvals_test < thresh).sum()} SNPs")

# genomic inflation factor
chi2_observed = stats.chi2.isf(pvals_test, df=1)
median_chi2_observed = np.median(chi2_observed)
median_chi2_expected = stats.chi2.ppf(0.5, df=1)
lambda_gc = median_chi2_observed / median_chi2_expected
print(f"\nGenomic inflation factor (λ): {lambda_gc:.4f}")

Single test iteration...
Done in 30.8s
p < 0.01: 2884 SNPs
p < 0.001: 432 SNPs
p < 0.0001: 75 SNPs

Genomic inflation factor (λ): 1.6169


In [3]:
from sklearn.decomposition import PCA
from scipy import stats
import numpy as np

def compute_lambda(pvals):
    chi2_observed = stats.chi2.isf(pvals, df=1)
    median_chi2_observed = np.median(chi2_observed)
    median_chi2_expected = stats.chi2.ppf(0.5, df=1)
    return median_chi2_observed / median_chi2_expected

for n_pcs in [10, 15, 20, 30]:
    pca_test = PCA(n_components=n_pcs, random_state=42, svd_solver='full')
    pcs_test = pca_test.fit_transform(X_standardized)

    age_col = X_confounders[:, 10]
    gender_col = X_confounders[:, 11]
    X_conf_test = np.hstack([pcs_test, age_col.reshape(-1,1), gender_col.reshape(-1,1)])

    _, pvals_n = doubleml_scan(X_standardized, Y, X_conf_test, random_state=0)
    lam = compute_lambda(pvals_n)
    n_sig = (pvals_n < 0.001).sum()
    print(f"n_PCs={n_pcs}: lambda={lam:.4f}, SNPs p<0.001={n_sig}")

n_PCs=10: lambda=1.6169, SNPs p<0.001=432
n_PCs=15: lambda=2.0220, SNPs p<0.001=7893
n_PCs=20: lambda=2.7764, SNPs p<0.001=20240
n_PCs=30: lambda=3.5908, SNPs p<0.001=24931


In [5]:
import numpy as np
import networkx as nx

threshold_test = 0.3

np.random.seed(0)
chosen_idx2 = np.random.RandomState(0).choice(X_standardized.shape[1], 5000, replace=False)
X_sub2 = X_standardized[:, chosen_idx2]
sample_corr = np.corrcoef(X_sub2)
np.fill_diagonal(sample_corr, 0)
del X_sub2

iu = np.triu_indices_from(sample_corr, k=1)
rows, cols = iu
strong = sample_corr[iu] > threshold_test
pairs_03 = [(sample_ids[rows[k]], sample_ids[cols[k]]) for k in range(len(strong)) if strong[k]]
print(f"Pairs above {threshold_test}: {len(pairs_03)}")

G2 = nx.Graph()
G2.add_nodes_from(sample_ids)
G2.add_edges_from(pairs_03)
clusters2 = [c for c in nx.connected_components(G2) if len(c) > 1]
samples_to_drop_03 = []
for cluster in clusters2:
    sorted_c = sorted(cluster)
    samples_to_drop_03.extend(sorted_c[1:])

print(f"Would drop {len(samples_to_drop_03)} additional samples at threshold 0.3")

keep_ids_03 = [s for s in sample_ids if s not in set(samples_to_drop_03)]
keep_idx_03 = [i for i, s in enumerate(sample_ids) if s in set(keep_ids_03)]

X_test03 = X_standardized[keep_idx_03]
Y_test03 = Y[keep_idx_03]

from sklearn.decomposition import PCA
from scipy import stats

def compute_lambda(pvals):
    chi2_observed = stats.chi2.isf(pvals, df=1)
    median_chi2_observed = np.median(chi2_observed)
    median_chi2_expected = stats.chi2.ppf(0.5, df=1)
    return median_chi2_observed / median_chi2_expected

for n_pcs in [10, 20]:
    pca_t = PCA(n_components=n_pcs, random_state=42, svd_solver='full')
    pcs_t = pca_t.fit_transform(X_test03)
    age_t = X_confounders[keep_idx_03, 10]
    gender_t = X_confounders[keep_idx_03, 11]
    Xc_t = np.hstack([pcs_t, age_t.reshape(-1,1), gender_t.reshape(-1,1)])
    _, pv_t = doubleml_scan(X_test03, Y_test03, Xc_t, random_state=0)
    lam_t = compute_lambda(pv_t)
    print(f"n_PCs={n_pcs}, n_samples={len(keep_idx_03)}: lambda={lam_t:.4f}")

Pairs above 0.3: 52
Would drop 51 additional samples at threshold 0.3
n_PCs=10, n_samples=1409: lambda=2.3078
n_PCs=20, n_samples=1409: lambda=2.3162


In [6]:
import numpy as np

def compute_lambda(pvals):
    from scipy import stats
    chi2_observed = stats.chi2.isf(pvals, df=1)
    median_chi2_observed = np.median(chi2_observed)
    median_chi2_expected = stats.chi2.ppf(0.5, df=1)
    return median_chi2_observed / median_chi2_expected

# average lambda over 5 repeats, on the CURRENT 1460-sample (0.5-threshold) dataset
lambdas = []
for seed in range(5):
    _, pv = doubleml_scan(X_standardized, Y, X_confounders, random_state=seed)
    lam = compute_lambda(pv)
    lambdas.append(lam)
    print(f"seed={seed}: lambda={lam:.4f}")

print(f"\nMean lambda (5 repeats): {np.mean(lambdas):.4f}")
print(f"Std across repeats: {np.std(lambdas):.4f}")

seed=0: lambda=1.6169
seed=1: lambda=1.7505
seed=2: lambda=1.6149
seed=3: lambda=1.5933
seed=4: lambda=1.7263

Mean lambda (5 repeats): 1.6604
Std across repeats: 0.0647


In [7]:
import numpy as np

def mean_lambda(X, Y_arr, X_conf, n_seeds=5):
    lambdas = []
    for seed in range(n_seeds):
        _, pv = doubleml_scan(X, Y_arr, X_conf, random_state=seed)
        lambdas.append(compute_lambda(pv))
    return np.mean(lambdas), np.std(lambdas)

# Test A: current data (1460 samples), but more PCs, averaged properly
from sklearn.decomposition import PCA
for n_pcs in [10, 20]:
    pca_t = PCA(n_components=n_pcs, random_state=42, svd_solver='full')
    pcs_t = pca_t.fit_transform(X_standardized)
    age_c = X_confounders[:, 10]
    gender_c = X_confounders[:, 11]
    Xc_t = np.hstack([pcs_t, age_c.reshape(-1,1), gender_c.reshape(-1,1)])
    mean_lam, std_lam = mean_lambda(X_standardized, Y, Xc_t, n_seeds=5)
    print(f"[1460 samples] n_PCs={n_pcs}: mean_lambda={mean_lam:.4f} (+/- {std_lam:.4f})")

# Test B: stricter relatedness threshold (1409 samples), 10 PCs, averaged properly
mean_lam_03, std_lam_03 = mean_lambda(X_test03, Y_test03, X_confounders[keep_idx_03][:, :12], n_seeds=5)
print(f"\n[1409 samples, threshold 0.3] n_PCs=10: mean_lambda={mean_lam_03:.4f} (+/- {std_lam_03:.4f})")

[1460 samples] n_PCs=10: mean_lambda=1.6604 (+/- 0.0647)
[1460 samples] n_PCs=20: mean_lambda=2.5501 (+/- 0.1584)

[1409 samples, threshold 0.3] n_PCs=10: mean_lambda=1.9944 (+/- 0.2200)


In [8]:
import numpy as np
import time

n_repeats = 30
threshold = 0.001
n_snps = X_standardized.shape[1]

significant_counts = np.zeros(n_snps, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_scan(X_standardized, Y, X_confounders, random_state=rep)
    significant_counts += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats} repeats, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")

stability_fraction = significant_counts / n_repeats
print("\nStability distribution:")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction >= t).sum()} SNPs")

Completed 5/30 repeats, elapsed 97.8s
Completed 10/30 repeats, elapsed 190.5s
Completed 15/30 repeats, elapsed 283.1s
Completed 20/30 repeats, elapsed 375.3s
Completed 25/30 repeats, elapsed 468.0s
Completed 30/30 repeats, elapsed 560.7s
Total time: 560.7s

Stability distribution:
  >= 50%: 1619 SNPs
  >= 60%: 959 SNPs
  >= 70%: 658 SNPs
  >= 80%: 345 SNPs
  >= 90%: 95 SNPs
  >= 100%: 22 SNPs


In [9]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"

stability_df = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
})
stability_df = stability_df.sort_values("stability_fraction", ascending=False)
stability_df.to_csv(os.path.join(out_dir, "checkpoint9_doubleml_stability_results_relatedness_filtered.csv"), index=False)
print("Saved full stability results.")

shortlist_80 = stability_df[stability_df["stability_fraction"] >= 0.8].copy()
print("Primary shortlist (>=80%):", len(shortlist_80))
shortlist_80.to_csv(os.path.join(out_dir, "shortlist_smoking_80pct_primary.csv"), index=False)

shortlist_100 = stability_df[stability_df["stability_fraction"] >= 1.0].copy()
print("Sensitivity shortlist (=100%):", len(shortlist_100))
shortlist_100.to_csv(os.path.join(out_dir, "shortlist_smoking_100pct_sensitivity.csv"), index=False)

print("\nSaved both.")

Saved full stability results.
Primary shortlist (>=80%): 345
Sensitivity shortlist (=100%): 22

Saved both.


In [10]:
import pandas as pd
import re

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
position_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

shortlist_80["core_name"] = shortlist_80["probe_id"].map(strip_address_suffix)
shortlist_80_pos = shortlist_80.merge(position_lookup, left_on="core_name", right_index=True, how="left")
print("Primary shortlist unresolved positions:", shortlist_80_pos["Chr"].isna().sum())

shortlist_100["core_name"] = shortlist_100["probe_id"].map(strip_address_suffix)
shortlist_100_pos = shortlist_100.merge(position_lookup, left_on="core_name", right_index=True, how="left")
print("Sensitivity shortlist unresolved positions:", shortlist_100_pos["Chr"].isna().sum())

Primary shortlist unresolved positions: 0
Sensitivity shortlist unresolved positions: 0


In [11]:
import numpy as np

probe_id_to_idx = {pid: i for i, pid in enumerate(probe_ids_valid)}

def get_genotype_vector(probe_id):
    return X_standardized[:, probe_id_to_idx[probe_id]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_genotype_vector(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_genotype_vector(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained_80 = greedy_ld_prune(shortlist_80_pos)
shortlist_80_pruned = shortlist_80_pos[shortlist_80_pos["probe_id"].isin(retained_80)].copy()
shortlist_80_pruned = shortlist_80_pruned.sort_values(["Chr", "MapInfo"]).reset_index(drop=True)
print("Primary (80%) after LD pruning:", len(shortlist_80_pruned))

Primary (80%) after LD pruning: 289


In [12]:
retained_100 = greedy_ld_prune(shortlist_100_pos)
shortlist_100_pruned = shortlist_100_pos[shortlist_100_pos["probe_id"].isin(retained_100)].copy()
shortlist_100_pruned = shortlist_100_pruned.sort_values(["Chr", "MapInfo"]).reset_index(drop=True)
print("Sensitivity (100%) after LD pruning:", len(shortlist_100_pruned))

Sensitivity (100%) after LD pruning: 18


In [13]:
import numpy as np
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"

def check_perfect_corr(shortlist_pruned_df):
    ids = shortlist_pruned_df["probe_id"].tolist()
    X_check = np.column_stack([get_genotype_vector(pid) for pid in ids])
    corr_matrix = np.corrcoef(X_check.T)
    to_remove = set()
    for i in range(len(ids)):
        for j in range(i+1, len(ids)):
            if abs(corr_matrix[i,j]) > 0.99 and ids[j] not in to_remove:
                to_remove.add(ids[j])
    return to_remove

to_remove_80 = check_perfect_corr(shortlist_80_pruned)
shortlist_80_final = shortlist_80_pruned[~shortlist_80_pruned["probe_id"].isin(to_remove_80)].copy()
print("Primary (80%) final:", len(shortlist_80_final), "| removed for perfect corr:", len(to_remove_80))

to_remove_100 = check_perfect_corr(shortlist_100_pruned)
shortlist_100_final = shortlist_100_pruned[~shortlist_100_pruned["probe_id"].isin(to_remove_100)].copy()
print("Sensitivity (100%) final:", len(shortlist_100_final), "| removed for perfect corr:", len(to_remove_100))

shortlist_80_final.to_csv(os.path.join(out_dir, "shortlist_smoking_80pct_final.csv"), index=False)
shortlist_100_final.to_csv(os.path.join(out_dir, "shortlist_smoking_100pct_final.csv"), index=False)
print("\nSaved both final shortlists.")

Primary (80%) final: 170 | removed for perfect corr: 119
Sensitivity (100%) final: 18 | removed for perfect corr: 0

Saved both final shortlists.


In [14]:
import numpy as np
import pandas as pd

# recompute correlation matrix on the 289 pre-removal SNPs to inspect the actual pairs
ids_289 = shortlist_80_pruned["probe_id"].tolist()
X_check_289 = np.column_stack([get_genotype_vector(pid) for pid in ids_289])
corr_matrix_289 = np.corrcoef(X_check_289.T)

# find all pairs above 0.99, with their positions
pairs_info = []
for i in range(len(ids_289)):
    for j in range(i+1, len(ids_289)):
        if abs(corr_matrix_289[i,j]) > 0.99:
            row_i = shortlist_80_pruned[shortlist_80_pruned["probe_id"] == ids_289[i]].iloc[0]
            row_j = shortlist_80_pruned[shortlist_80_pruned["probe_id"] == ids_289[j]].iloc[0]
            dist = abs(row_i["MapInfo"] - row_j["MapInfo"]) if row_i["Chr"] == row_j["Chr"] else None
            pairs_info.append({
                "snp_i": ids_289[i], "snp_j": ids_289[j], "r": corr_matrix_289[i,j],
                "same_chr": row_i["Chr"] == row_j["Chr"], "distance_bp": dist
            })

pairs_df = pd.DataFrame(pairs_info)
print("Total near-perfect pairs:", len(pairs_df))
print("\nSame chromosome:", pairs_df["same_chr"].sum())
print("Distance distribution (bp) for same-chr pairs:")
print(pairs_df.loc[pairs_df["same_chr"], "distance_bp"].describe())

# MAF of the affected SNPs
affected_ids = set(pairs_df["snp_i"]) | set(pairs_df["snp_j"])
maf_check = []
for pid in affected_ids:
    g = get_genotype_vector(pid)
    freq = g.mean()  # standardized already, so use raw genotype instead - recompute from encoded data if needed
    maf_check.append(freq)
print(f"\nNumber of unique SNPs involved in near-perfect pairs: {len(affected_ids)}")

Total near-perfect pairs: 6339

Same chromosome: 316
Distance distribution (bp) for same-chr pairs:
count    3.160000e+02
mean     6.415138e+07
std      5.238157e+07
min      1.243035e+06
25%      2.191771e+07
50%      5.159489e+07
75%      9.071975e+07
max      2.453650e+08
Name: distance_bp, dtype: float64

Number of unique SNPs involved in near-perfect pairs: 124
